In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.inspection import permutation_importance
import matplotlib.pyplot as plt

# Load dataset
df = pd.read_csv('dataset_thesis_new_hdi_v8.csv')
# --- FIX RUS AND BLR 2021 TEAM SIZE BEFORE LAGGING ---
# We force the 2021 Team Size so that when we shift(1) later, 
# the 2024 Lagged_Team_Size for these countries is historically accurate.
df.loc[(df['iso_code_mapped'] == 'RUS') & (df['year'] == 2021), 'Team_Size'] = 335
df.loc[(df['iso_code_mapped'] == 'BLR') & (df['year'] == 2021), 'Team_Size'] = 101

df = df.sort_values(['iso_code_mapped', 'year'])

#  Create t-1 Lags 
for col in ['GDP_merged', 'population', 'hdi', 'gdi', 'perc_athletic_prime']:
    df[f'{col}_t_1'] = df.groupby('iso_code_mapped')[col].shift(1)

#  GDP growth cycle
df['GDP_t_4'] = df.groupby('iso_code_mapped')['GDP_merged'].shift(4)
df['GDP_Growth_Cycle'] = ((df['GDP_merged_t_1'] - df['GDP_t_4']) / df['GDP_t_4']) * 100

#  Filter to Olympic years and shift
olympic_years = [1996, 2000, 2004, 2008, 2012, 2016, 2021, 2024]
oly = df[df['year'].isin(olympic_years)].copy()
oly = oly.sort_values(['iso_code_mapped', 'year'])

oly['Lagged_Team_Size'] = oly.groupby('iso_code_mapped')['Team_Size'].shift(1)
oly['Prev_Total_Medals'] = oly.groupby('iso_code_mapped')['Total'].shift(1)

# Target variable: Medal Share %
global_medals = oly.groupby('year')['Total'].sum().rename('Global_Total')
oly = oly.merge(global_medals, on='year')
oly['Medal_Share_Pct'] = (oly['Total'] / oly['Global_Total']) * 100

oly['Log_GDP_t_1'] = np.log10(oly['GDP_merged_t_1'])
oly['Log_Pop_t_1'] = np.log10(oly['population_t_1'])

# Impute & Clean
final_features = [
    'Log_GDP_t_1', 'Log_Pop_t_1', 'GDP_Growth_Cycle', 
    'perc_athletic_prime_t_1', 'Lagged_Team_Size', 'Prev_Total_Medals', 
    'hdi_t_1', 'gdi_t_1', 'Is_Host'
]
for f in final_features:
    if f in ['Lagged_Team_Size', 'Prev_Total_Medals']:
        oly[f] = oly[f].fillna(0)
    else:
        oly[f] = oly[f].fillna(oly[f].mean())

oly_clean = oly.dropna(subset=['Medal_Share_Pct', 'Log_GDP_t_1']).copy()

#  Train/Test Split (< 2024 vs == 2024)
train = oly_clean[oly_clean['year'] < 2024]
test = oly_clean[oly_clean['year'] == 2024]

X_train = train[final_features]
y_train = train['Medal_Share_Pct']
X_test = test[final_features]
y_test = test['Medal_Share_Pct']

# Train RF
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

# Predict and build the table
test_pred = rf.predict(X_test)
test_results = test.copy()
test_results['Predicted_Share_Pct'] = test_pred
total_2024_medals = test_results['Global_Total'].iloc[0]
test_results['Predicted_Total_Medals'] = (test_pred / 100) * total_2024_medals

# Evaluate
print(f"R2 Score (2024): {r2_score(y_test, test_pred):.2f}")
print(f"MAE (2024): {mean_absolute_error(y_test, test_pred):.3f}%\n")

comp_table = test_results[['iso_code_mapped', 'Total', 'Predicted_Total_Medals']].copy()
comp_table.columns = ['Country', 'Actual_Medals_2024', 'Predicted_Medals_2024']
comp_table['Predicted_Medals_2024'] = comp_table['Predicted_Medals_2024'].round(1)
comp_table = comp_table.sort_values('Actual_Medals_2024', ascending=False).head(20)

print(comp_table.to_string(index=False))
comp_table.to_csv('2024_actual_vs_predicted.csv', index=False)

# Permutation Importance
perm = permutation_importance(rf, X_test, y_test, n_repeats=10, random_state=42, n_jobs=-1)
imp = pd.Series(perm.importances_mean, index=final_features).sort_values(ascending=False)
print("\nFeature Importance (2024 Test Set):")
print(imp)

R2 Score (2024): 0.80
MAE (2024): 0.186%

Country  Actual_Medals_2024  Predicted_Medals_2024
    USA                 126                  109.2
    CHN                  91                   92.1
    GBR                  65                   65.7
    FRA                  64                   48.5
    AUS                  53                   43.1
    JPN                  45                   61.3
    ITA                  40                   42.9
    NLD                  34                   35.4
    DEU                  33                   37.9
    KOR                  32                   13.0
    CAN                  27                   12.7
    BRA                  20                   10.1
    NZL                  20                   10.3
    HUN                  19                   11.7
    ESP                  18                   11.0
    UZB                  13                    3.6
    IRN                  12                    6.2
    UKR                  12             

In [1]:
#OTHER PREDICTION OF 2024 BUT NOW WITH TEAM SIZE
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.inspection import permutation_importance

#  Load Dataset
df = pd.read_csv('dataset_thesis_new_hdi_v8.csv')

#  Fix Team Sizes for 2021 (Infrastructural Capacity)

team_2021_overrides = {
    'RUS': 335,
    'BLR': 101
}
for iso, size in team_2021_overrides.items():
    df.loc[(df['iso_code_mapped'] == iso) & (df['year'] == 2021), 'Team_Size'] = size

#  Data Cleaning & Bridge 
# Carry data forward through non-Olympic years
df = df.sort_values(['iso_code_mapped', 'year'])
cols_to_ffill = ['GDP_merged', 'population', 'hdi', 'gdi', 'perc_athletic_prime', 'Team_Size']
for col in cols_to_ffill:
    df[col] = df.groupby('iso_code_mapped')[col].ffill()

#  Create Predictive Lags (t-1)

for col in ['GDP_merged', 'population', 'hdi', 'gdi', 'perc_athletic_prime']:
    df[f'{col}_t_1'] = df.groupby('iso_code_mapped')[col].shift(1)

#  Filter for Olympic Years to create cyclical lags (t-4)
olympic_years = [1996, 2000, 2004, 2008, 2012, 2016, 2021, 2024]
oly = df[df['year'].isin(olympic_years)].copy()
oly = oly.sort_values(['iso_code_mapped', 'year'])

# Shifting medals and team size by 1 Olympic cycle
oly['Prev_Total_Medals'] = oly.groupby('iso_code_mapped')['Total'].shift(1)
oly['Lagged_Team_Size'] = oly.groupby('iso_code_mapped')['Team_Size'].shift(1)


oly['Log_GDP_t_1'] = np.log10(oly['GDP_merged_t_1'])
oly['Log_Pop_t_1'] = np.log10(oly['population_t_1'])


features = [
    'Log_GDP_t_1', 'Log_Pop_t_1', 'perc_athletic_prime_t_1', 
    'Lagged_Team_Size', 'Prev_Total_Medals', 'hdi_t_1', 'gdi_t_1', 'Is_Host'
]

# Impute remaining NaNs 
for f in features:
    if f in ['Prev_Total_Medals', 'Lagged_Team_Size']:
        oly[f] = oly[f].fillna(0)
    else:
        oly[f] = oly[f].fillna(oly[f].mean())

#  Train/Test Split (Train < 2024, Test == 2024)
train = oly[(oly['year'] < 2024) & oly['Total'].notna()]
test = oly[oly['year'] == 2024].copy()

X_train = train[features]
y_train = train['Total']
X_test = test[features]
y_test = test['Total']

#  Train Random Forest
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

#  Predict and Evaluate
y_pred = rf.predict(X_test)
r2_val = r2_score(y_test, y_pred)
mae_val = mean_absolute_error(y_test, y_pred)

print(f"R2 Score (Paris 2024): {r2_val:.4f}")
print(f"MAE (Paris 2024): {mae_val:.2f} medals\n")

# Feature Importance (Permutation)
perm = permutation_importance(rf, X_test, y_test, n_repeats=30, random_state=42, n_jobs=-1)
imp_df = pd.DataFrame({
    'Feature': features,
    'Importance': perm.importances_mean,
    'Std': perm.importances_std
}).sort_values('Importance', ascending=False)

print("Feature Importance:")
print(imp_df.to_string(index=False))

# Actual vs Predicted Table (Top 20)
test['Predicted'] = y_pred.round(1)
comparison = test[['iso_code_mapped', 'Total', 'Predicted']].sort_values('Total', ascending=False).head(20)
comparison.columns = ['Country', 'Actual_Medals', 'Predicted_Medals']

print("\nComparison Table (Top 20 Nations in Paris 2024):")
print(comparison.to_string(index=False))

R2 Score (Paris 2024): 0.8226
MAE (Paris 2024): 1.68 medals

Feature Importance:
                Feature  Importance      Std
      Prev_Total_Medals    1.203869 0.112692
                Is_Host    0.071944 0.022503
       Lagged_Team_Size    0.031432 0.003072
            Log_Pop_t_1    0.024846 0.009440
            Log_GDP_t_1    0.012026 0.001014
                hdi_t_1    0.008204 0.002631
perc_athletic_prime_t_1    0.004222 0.002393
                gdi_t_1   -0.012386 0.010820

Comparison Table (Top 20 Nations in Paris 2024):
Country  Actual_Medals  Predicted_Medals
    USA            126             115.5
    CHN             91              95.5
    GBR             65              62.1
    FRA             64              47.0
    AUS             53              41.0
    JPN             45              56.2
    ITA             40              41.5
    NLD             34              35.9
    DEU             33              40.1
    KOR             32              23.3
    CAN      